# Three measurements lift a plane
### From incidence, to cardinality, to motion

Can measurements of three different solids move a separate plane into a simple shape?

We will count vertical columns, keep the counts as arrangements, and use their
values as displacements. The construction code stays beside the mathematical
description. Plotting helpers consume captured results only.

This is a continuation of [Three incidences fill a box](03_three_incidence_box.ipynb),
but all definitions are included here. Use the **Kaleion** kernel and **Run All**.
See the [lesson notes](../docs/lessons/04_measured_motion.md) for the narrative and
[setup instructions](README.md) for installation and offline viewing.

In [ ]:
from pathlib import Path
from math import gcd, prod
from itertools import combinations
import json
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
from kaleion import Collection, F, Motion, Workspace, choose, param, vector
from kaleion.viewers.plotly import animation_figure

pio.renderers.default = "plotly_mimetype+notebook"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))
from snapshot_views import keyed_values, rectangular_values

OUTPUT = ROOT / "build" / "notebooks" / "measured-motion"
OUTPUT.mkdir(parents=True, exist_ok=True)
PARAMETERS = {"a": 11, "b": 7, "c": 5}
assert all(isinstance(v, int) and not isinstance(v, bool) and v > 1
           for v in PARAMETERS.values())
assert prod(v - 1 for v in PARAMETERS.values()) <= 1500, "Use a small visible box."
a, b, c = param("a"), param("b"), param("c")
COLORS = {"X": "#49c5b6", "Y": "#f0bc63", "Z": "#a5a0ff"}
BG, INK = "#101b2b", "#e8eef7"

## 1 · Construct three incidences on one box

Let $D=\{1,\ldots,a-1\}\times\{1,\ldots,b-1\}\times\{1,\ldots,c-1\}$.
The incidence $X$ selects points where $x/a$ is greatest; $Y$ and $Z$ do the
same for $y/b$ and $z/c$. Ties belong to every qualifying incidence.

We compare integer cross-products, so incidence never depends on rounded coordinates.
The attributes `u, v, w` retain the integer coordinates independently of placement.

In [ ]:
box = (Collection.grid(a - 1, b - 1, c - 1, values=1)
       .annotate(u=F.i + 1, v=F.j + 1, w=F.k + 1)
       .arrange(F.u, F.v, F.w))
X = box.where((a * F.v <= b * F.u) & (a * F.w <= c * F.u))
Y = box.where((b * F.u <= a * F.v) & (b * F.w <= c * F.v))
Z = box.where((c * F.u <= a * F.w) & (c * F.v <= b * F.w))

## 2 · Collapse columns into measured arrangements

\[
h_X(x,y)=\#\{z:(x,y,z)\in X\},
\qquad h_Y(x,y),\ h_Z(x,y)\text{ similarly}.
\]

`count(by=(F.u, F.v))` **retains** $(x,y)$ and sums over $z$.
All three measurements live on the same key domain, including zero counts.
Orthogonal projections would have different key domains and need a declared
correspondence before being combined.

In [ ]:
h_X = X.count(by=(F.u, F.v)).arrange(F.u, F.v)
h_Y = Y.count(by=(F.u, F.v)).arrange(F.u, F.v)
h_Z = Z.count(by=(F.u, F.v)).arrange(F.u, F.v)

dx = h_X.bind(on=(F.u, F.v), key=(F.u, F.v))
dy = h_Y.bind(on=(F.u, F.v), key=(F.u, F.v))
dz = h_Z.bind(on=(F.u, F.v), key=(F.u, F.v))
total = h_X.with_values(F.value + dy + dz)
discrepancy = total.with_values(F.value - (c - 1))

# This plane has its own occurrences; a measured value drives each matching key.
plane = (Collection.grid(a - 1, b - 1, values=0)
         .annotate(u=F.i + 1, v=F.j + 1).arrange(F.u, F.v, 0))
after_X = plane.with_values(F.value + dx).move(vector(0, 0, dx))
after_Y = after_X.with_values(F.value + dy).move(vector(0, 0, dy))
after_Z = after_Y.with_values(F.value + dz).move(vector(0, 0, dz))

ROOTS = {"D": box, "X": X, "Y": Y, "Z": Z,
         "h_X": h_X, "h_Y": h_Y, "h_Z": h_Z,
         "total": total, "discrepancy": discrepancy,
         "plane": plane, "lifted": after_Z}
workspace = Workspace(ROOTS, PARAMETERS)
assert not workspace.state.errors, dict(workspace.state.errors)
state = workspace.state

In [ ]:
# The shared snapshot adapters validate keys; these helpers own Plotly styling.

def height_maps(captured):
    fig = make_subplots(rows=1, cols=3, subplot_titles=["h_X · X columns", "h_Y · Y columns", "h_Z · Z columns"])
    for col, name in enumerate(("X", "Y", "Z"), 1):
        xs, ys, values = rectangular_values(captured.results["h_" + name], x="u", y="v")
        fig.add_trace(go.Heatmap(
            x=xs, y=ys, z=values, zmin=0, zmax=captured.parameters["c"] - 1,
            colorscale=[[0, BG], [1, COLORS[name]]], showscale=False,
            text=[[str(v) for v in row] for row in values], texttemplate="%{text}",
            hovertemplate="(x,y)=(%{x},%{y})<br>column count=%{text}<extra></extra>",
            xgap=2, ygap=2), row=1, col=col)
        fig.update_xaxes(title_text="x · retained key", dtick=1, row=1, col=col)
        fig.update_yaxes(title_text="y", dtick=1, row=1, col=col)
    fig.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
                      font=dict(color=INK), height=440, margin=dict(t=100, b=60),
                      title="Three arrangements of measured cardinalities")
    return fig

height_plot = height_maps(state)
height_plot.show()

## 3 · Let the measurements move another arrangement

Each step adds one measured field to the plane's height and integer label:

\[
0\longrightarrow h_X\longrightarrow h_X+h_Y\longrightarrow h_X+h_Y+h_Z.
\]

Press **Play**, or scrub between the stages. Can you predict the final shape?
The faint reference plane is at $z=c-1$. Grid edges connect neighboring retained
keys for orientation; they do not add mathematical points or define a continuous solid.
Undo traverses the exact captured paths in reverse. Intermediate heights are presentation only.

In [ ]:
motion_workspace = Workspace({"plane": plane}, PARAMETERS)
forward = [motion_workspace.set("plane", stage, motion=Motion())
           for stage in (after_X, after_Y, after_Z)]
lifted_capture = motion_workspace.capture("Three measured column fields have moved this independent plane.")
backward = [motion_workspace.undo() for _ in forward]

samples, labels = [], []
for name, transition in zip(("add h_X", "add h_Y", "add h_Z", "undo h_Z", "undo h_Y", "undo h_X"),
                            forward + backward):
    for t in np.linspace(0, 1, 13):
        samples.append(transition.frame("plane", float(t)))
        labels.append(f"{name} · {t:.0%}")

assert state.results["plane"].ids == state.results["lifted"].ids
assert samples[0].before_ids == samples[-1].before_ids
np.testing.assert_array_equal(samples[0].positions, samples[-1].positions)
for go_forward, go_back in zip(forward, reversed(backward)):
    np.testing.assert_array_equal(go_forward.frame("plane", .25).positions,
                                  go_back.frame("plane", .75).positions)

In [ ]:
def plane_motion(captured, frames, captions):
    source = captured.results["plane"]
    keys = list(zip(map(int, source.fields["u"]), map(int, source.fields["v"])))
    by_key = dict(zip(keys, source.ids))
    edges = [(by_key[key], by_key[neighbor]) for key in keys
             for neighbor in ((key[0] + 1, key[1]), (key[0], key[1] + 1)) if neighbor in by_key]

    def grid(frame):
        assert frame.before_ids == frame.after_ids
        positions = dict(zip(frame.before_ids, frame.positions))
        xyz = [[v for a, b in edges for v in (positions[a][axis], positions[b][axis], None)]
               for axis in range(3)]
        return go.Scatter3d(x=xyz[0], y=xyz[1], z=xyz[2], mode="lines",
                            line=dict(color="#52738b", width=2), hoverinfo="skip", showlegend=False)

    fig = animation_figure(frames, labels=captions, title="Three counts lift an independent plane",
                           duration=65, show_values=False)
    fig.add_trace(grid(frames[0]))
    h = captured.parameters["c"] - 1
    xmax, ymax = captured.parameters["a"] - 1, captured.parameters["b"] - 1
    fig.add_trace(go.Surface(x=[.5, xmax + .5], y=[.5, ymax + .5], z=[[h, h], [h, h]],
                             opacity=.12, showscale=False, colorscale=[[0, "#8ca9ba"], [1, "#8ca9ba"]],
                             hoverinfo="skip"))
    for i, frame in enumerate(fig.frames):
        frame.data = tuple(frame.data) + (grid(frames[i]),)
        frame.traces = (0, 1)
    fig.update_layout(scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="measured height",
                                  zaxis=dict(range=[-.5, max(h, max(float(f.positions[:, 2].max()) for f in frames)) + .8])))
    return fig

lift_plot = plane_motion(state, samples, labels)
lift_plot.show()

## 4 · Explain the flatness, and its quotient structure

For **pairwise coprime** $a,b,c$, no two normalized coordinates can tie at an
interior point. For example, $bx=ay$ and $\gcd(a,b)=1$ would require $a\mid x$,
which is impossible for $1\le x<a$. Every voxel therefore belongs to exactly one
incidence. Every column contains $c-1$ voxels, so **pointwise**

\[
\boxed{h_X(x,y)+h_Y(x,y)+h_Z(x,y)=c-1.}
\]

This argument proves the general statement under the stated assumptions. The code
below checks particular finite cases and witnesses; floating animation is not the proof.

The upper part of a column belongs to $Z$. Its two thresholds yield

\[
h_Z(x,y)=c-1-\max\!\left(\left\lfloor cx/a\right\rfloor,
                                \left\lfloor cy/b\right\rfloor\right).
\]

The strict lower part belongs to $X$ if $bx>ay$, or to $Y$ if $ay>bx$.
Thus the measured surfaces also display a relationship between two quotient profiles.

In [ ]:
def inspect_case(captured):
    assert not captured.errors, dict(captured.errors)
    r = captured.results
    av, bv, cv = (captured.parameters[k] for k in ("a", "b", "c"))
    maps = [keyed_values(r["h_" + name], keys=("u", "v")) for name in ("X", "Y", "Z")]
    total = keyed_values(r["total"], keys=("u", "v"))
    discrepancy = keyed_values(r["discrepancy"], keys=("u", "v"))
    expected_keys = {(x, y) for x in range(1, av) for y in range(1, bv)}
    assert all(set(m) == expected_keys for m in maps)
    assert set(total) == expected_keys == set(discrepancy)
    assert r["plane"].ids == r["lifted"].ids
    np.testing.assert_array_equal(r["lifted"].positions[:, 2], r["lifted"].values)
    is_coprime = all(gcd(x, y) == 1 for x, y in combinations((av, bv, cv), 2))
    if is_coprime:
        assert set(discrepancy.values()) == {0}
        assert all(maps[2][x, y] == cv - 1 - max(cv*x//av, cv*y//bv) for x, y in expected_keys)
    return {"parameters": dict(captured.parameters), "pairwise_coprime": is_coprime,
            "height_levels": sorted(set(total.values())),
            "witnesses": [{"key": list(k), "excess": value}
                          for k, value in discrepancy.items() if value]}

report = inspect_case(state)
print(json.dumps(report, indent=2))

## 5 · A bump is a counterexample you can inspect

Change the parameters to $(6,4,5)$. Their joint gcd is 1, but $\gcd(6,4)=2$.
At $(x,y)=(3,2)$, both $X$ and $Y$ count $z=1,2$, while $Z$ counts $z=3,4$.
The driver values are $(2,2,2)$, so this point reaches height $6$ instead of $4$.

The discrepancy arrangement stores $h_X+h_Y+h_Z-(c-1)$ as exact integers.
A zero discrepancy says these **keyed column totals** agree; it does not by itself
say two arbitrary arrangements contain the same occurrences.

In [ ]:
edit = workspace.set_parameters(a=6, b=4, c=5)
overlap_state = workspace.state
overlap_report = inspect_case(overlap_state)
assert overlap_report["witnesses"] == [{"key": [3, 2], "excess": 2}]
workspace.capture("At (3,2), X and Y count the same two voxels; the height excess is 2.")
workspace.undo()
assert workspace.state is state  # The captured mathematical state is restored.
print(json.dumps(overlap_report, indent=2))

In [ ]:
def discrepancy_figure(captured):
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]],
                        subplot_titles=["Measured endpoint · reference height 4", "Excess at each retained key"])
    s = captured.results["lifted"]
    excess = keyed_values(captured.results["discrepancy"], keys=("u", "v"))
    values = [excess[int(x), int(y)] for x, y in zip(s.fields["u"], s.fields["v"])]
    fig.add_trace(go.Scatter3d(x=s.positions[:, 0].tolist(), y=s.positions[:, 1].tolist(),
                               z=s.positions[:, 2].tolist(), mode="markers",
                               marker=dict(size=7, color=["#f083a5" if v else "#49c5b6" for v in values]),
                               text=[f"height={int(v)}; excess={e}" for v, e in zip(s.values, values)],
                               hovertemplate="(%{x},%{y})<br>%{text}<extra></extra>"), row=1, col=1)
    xs, ys, matrix = rectangular_values(captured.results["discrepancy"], x="u", y="v")
    fig.add_trace(go.Heatmap(x=xs, y=ys, z=matrix, zmin=0, zmax=2, showscale=False,
                             colorscale=[[0, BG], [1, "#f083a5"]], xgap=3, ygap=3,
                             text=[[str(v) for v in row] for row in matrix], texttemplate="%{text}",
                             hovertemplate="(%{x},%{y})<br>excess=%{text}<extra></extra>"), row=1, col=2)
    fig.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
                      title="Relax coprimality · a bump exposes double-counting", showlegend=False,
                      height=520, scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="height",
                                              zaxis=dict(range=[0, 7]), aspectmode="data"))
    fig.update_xaxes(title_text="x", dtick=1)
    fig.update_yaxes(title_text="y", dtick=1)
    return fig

overlap_plot = discrepancy_figure(overlap_state)
overlap_plot.show()

## 6 · Follow one movement back to its contributors

Set `COLUMN` below and rerun this cell to inspect another key. This small explanation
joins a target occurrence, the three bound count occurrences, and the exact original
contributors. It uses the declared correspondence, not proximity in the picture.

The operation graph already retains the driver dependency. This notebook makes the
key lookup explicit; Kaleion does not yet supply a general clickable explanation UI.

In [ ]:
COLUMN = (3, 2)

def explain_column(captured, key):
    r = captured.results
    domain = r["D"]
    coordinates = {oid: [int(domain.fields[n][i]) for n in ("u", "v", "w")]
                   for i, oid in enumerate(domain.ids)}
    target_keys = list(zip(map(int, r["lifted"].fields["u"]), map(int, r["lifted"].fields["v"])))
    target_index = target_keys.index(key)
    drivers = []
    for name in ("X", "Y", "Z"):
        h = r["h_" + name]
        keys = list(zip(map(int, h.fields["u"]), map(int, h.fields["v"])))
        index = keys.index(key)
        contributors = h.contributor_ids(key)
        assert len(contributors) == int(h.values[index])
        drivers.append({"name": name, "count": int(h.values[index]), "occurrence": h.ids[index],
                        "contributors": [{"occurrence": oid, "point": coordinates[oid]} for oid in contributors]})
    return {"key": list(key), "target_occurrence": r["lifted"].ids[target_index],
            "height": int(r["lifted"].values[target_index]), "drivers": drivers}

explanation = explain_column(overlap_state, COLUMN)
lines = ["| Driver | Count / displacement | Original z coordinates |", "| --- | ---: | --- |"]
for driver in explanation["drivers"]:
    lines.append(f"| h_{driver['name']} | {driver['count']} | "
                 + ", ".join(str(item["point"][2]) for item in driver["contributors"]) + " |")
display(Markdown("\n".join(lines)))

column_plot = go.Figure()
for driver in explanation["drivers"]:
    column_plot.add_trace(go.Scatter(
        x=[driver["name"]] * len(driver["contributors"]),
        y=[item["point"][2] for item in driver["contributors"]], mode="markers",
        marker=dict(size=24, color=COLORS[driver["name"]], symbol="square"),
        name=f"{driver['name']} · count {driver['count']}",
        hovertemplate=f"{driver['name']} contributor at z=%{{y}}<extra></extra>"))
column_plot.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG, height=420,
                          title=f"Column {COLUMN} · follow the bump back to original voxels",
                          xaxis=dict(title="Incidence", categoryorder="array", categoryarray=["X", "Y", "Z"]),
                          yaxis=dict(title="Original integer z", dtick=1, range=[.5, overlap_state.parameters["c"] - .5]))
column_plot.show()

## 7 · Keep the definitions, measurements, and reversible history

The saved measurement workspace retains its parameter edit and pending redo.
The motion workspace retains the three captured moves and their undo paths.
Contributor IDs survive save/reopen, including groups whose count is zero.

Try a different pairwise coprime triple, another column, or a different incidence
predicate. Rerun downstream cells after changing definitions. A scene control only
changes presentation; it does not silently edit the mathematical workspace.

In [ ]:
workspace.capture("Column counts reused as keyed motion drivers; the general partition proof is written separately.")
for name, fig in (("height-maps", height_plot), ("lift-and-undo", lift_plot),
                  ("discrepancy", overlap_plot), ("column-explanation", column_plot)):
    fig.write_html(OUTPUT / f"{name}.html", include_plotlyjs=True, full_html=True, auto_play=False)
for name, investigation in (("measurements", workspace), ("motion", motion_workspace)):
    payload = investigation.to_json()
    (OUTPUT / f"{name}-workspace.json").write_text(payload)
    restored = Workspace.from_json(payload)
    assert not restored.state.errors
    assert restored.can_redo
restored = Workspace.from_json(workspace.to_json())
for name in ("X", "Y", "Z"):
    original = state.results["h_" + name]
    reopened = restored.state.results["h_" + name]
    for key in keyed_values(original, keys=("u", "v")):
        assert original.contributor_ids(key) == reopened.contributor_ids(key)
(OUTPUT / "cases.json").write_text(json.dumps([report, overlap_report], indent=2))
(OUTPUT / "column-explanation.json").write_text(json.dumps(explanation, indent=2))
print("Saved four offline figures, two workspaces, finite cases, and a contributor explanation to", OUTPUT)

The reusable mathematical pattern is **count along fibers → align by keys → apply
the measured values to another arrangement → inspect the discrepancy**. Counting
along a projection is a pushforward of counting measure; reading the resulting
field through target keys is a pullback. These names describe the mathematics
without requiring additional software objects.

Next: [Can line counts recover an image?](05_finite_radon.ipynb)